# {{ cookiecutter.experiment_name }}: analysis

This notebook reads immutable run artifacts. It does not execute stages or pipelines. Add exact completed run IDs below and run it from the project root.


In [ ]:
from pathlib import Path

import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print

from retrieval_core.utils.analysis import (
    build_analysis_frames,
    load_metrics_frame,
    metric_comparison_table,
    plot_metric_comparison,
    plot_query_metric_comparison,
)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Run this notebook from the project root.")

INFERENCE_RUNS: dict[str, str] = {
    # "baseline": "{{ cookiecutter.experiment_slug }}--baseline",
    # "treatment": "{{ cookiecutter.experiment_slug }}--treatment",
}
EVALUATION_RUNS: dict[str, str] = {
    # "baseline": "{{ cookiecutter.experiment_slug }}--baseline-evaluation",
    # "treatment": "{{ cookiecutter.experiment_slug }}--treatment-evaluation",
}
BASELINE = "baseline"
TREATMENT = "treatment"
TARGET_METRICS = {{ cookiecutter.evaluation_metrics }}

## Experiment and hypothesis

{{ cookiecutter.experiment_short_description }}

{{ cookiecutter.hypothesis }}


## Runs and target metrics


In [ ]:
run_rows = [{"stage": stage, "label": label, "run_id": run_id} for stage, runs in [("inference", INFERENCE_RUNS), ("evaluation", EVALUATION_RUNS)] for label, run_id in runs.items()]
display(pd.DataFrame(run_rows, columns=["stage", "label", "run_id"]))

if EVALUATION_RUNS:
    metrics_df = load_metrics_frame(EVALUATION_RUNS, project_root=PROJECT_ROOT)
    comparison_df = metric_comparison_table(metrics_df, baseline=BASELINE, treatment=TREATMENT)
    comparison_df = comparison_df.loc[comparison_df["metric"].isin(TARGET_METRICS)]
    display(comparison_df)
    plot_metric_comparison(comparison_df, baseline=BASELINE, treatment=TREATMENT)
else:
    metrics_df = pd.DataFrame()
    comparison_df = pd.DataFrame()
    print("Add exact evaluation run IDs to render aggregate results.")

## Query-level diagnostics


In [ ]:
if INFERENCE_RUNS:
    predictions_df, query_summary_df, qrels_df = build_analysis_frames(INFERENCE_RUNS, project_root=PROJECT_ROOT)
    display(query_summary_df.head())
    plot_query_metric_comparison(query_summary_df, baseline=BASELINE, treatment=TREATMENT)
else:
    predictions_df = pd.DataFrame()
    query_summary_df = pd.DataFrame()
    qrels_df = pd.DataFrame()
    print("Add exact inference run IDs to render query-level diagnostics.")

## Conclusion

State whether the target metrics support the preregistered hypothesis and keep interpretation separate from the observed artifact values.
